# Final project: approach tests

These tests compare the options behind the choices in `Final Project.ipynb`, using only methods from the course. Run that notebook first, because this one reads its stored results. There is no ground truth for the robot's position, so the detector of the main notebook serves as the reference for the other detectors. Its boxes were checked by eye on a sample of frames.

#### Setup
The cell loads the stored calibration and detections, and repeats the small helper functions of the main notebook.

In [2]:
import glob
import os
import time

import cv2
import imutils
import numpy as np
from matplotlib import pyplot as plt
from sklearn import svm
from sklearn.decomposition import PCA

RESULTS = 'results'
SEQUENCES = ['1_no_obstacle', '2_obstacle', '3_curved_bend', '4_circle']
IMAGE_SIZE = (1280, 720)
BOARD = (7, 5)
MIN_AREA = 1000

stored = np.load(os.path.join(RESULTS, 'calibration.npz'))
K1, d1, K2, d2, R, T = stored['K1'], stored['d1'], stored['K2'], stored['d2'], stored['R'], stored['T']
R1, R2, P1, P2 = stored['R1'], stored['R2'], stored['P1'], stored['P2']
map_l = cv2.initUndistortRectifyMap(K1, d1, R1, P1, IMAGE_SIZE, cv2.CV_32FC1)
map_r = cv2.initUndistortRectifyMap(K2, d2, R2, P2, IMAGE_SIZE, cv2.CV_32FC1)
FOCAL, BASELINE = P1[0, 0], -P2[0, 3] / P2[0, 0]
valid = (map_l[0] >= 0) & (map_l[0] < IMAGE_SIZE[0]) & (map_l[1] >= 0) & (map_l[1] < IMAGE_SIZE[1])
LEFT_LIMIT = max(np.flatnonzero(valid[row])[0] for row in range(300, 650))
RIGHT_LIMIT = min(np.flatnonzero(valid[row])[-1] for row in range(300, 650))

detections = {s: np.load(os.path.join(RESULTS, s + '_detections.npy')) for s in SEQUENCES}
backgrounds = {s: cv2.imread(os.path.join(RESULTS, s + '_background_left.png')) for s in SEQUENCES}
backgrounds_r = {s: cv2.imread(os.path.join(RESULTS, s + '_background_right.png')) for s in SEQUENCES}
block_matcher = cv2.StereoBM_create(numDisparities=64, blockSize=15)


def image_files(sequence, side):
    return sorted(glob.glob(os.path.join(sequence, sequence, side, '*.png')))


def rectified(path, side, maps=None):
    if maps is None:
        maps = map_l if side == 'left' else map_r
    return cv2.remap(cv2.imread(path), maps[0], maps[1], cv2.INTER_LINEAR)


def foreground(image, background, threshold=18):
    difference = np.abs(cv2.GaussianBlur(image, (5, 5), 0).astype(int) - cv2.GaussianBlur(background, (5, 5), 0).astype(int)).max(axis=2)
    return cv2.dilate(cv2.erode((difference > threshold).astype(np.uint8) * 255, None, iterations=1), None, iterations=2)


def blob_list(mask):
    contours = imutils.grab_contours(cv2.findContours(mask.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE))
    blobs = []
    for c in contours:
        x0, x1, y0, y1 = c[:, :, 0].min(), c[:, :, 0].max() + 1, c[:, :, 1].min(), c[:, :, 1].max() + 1
        blobs.append((np.sum(mask[y0:y1, x0:x1] > 0), x0, y0, x1 - x0, y1 - y0))
    return sorted(blobs, reverse=True)


def iou(a, b):
    w = min(a[0] + a[2], b[0] + b[2]) - max(a[0], b[0])
    h = min(a[1] + a[3], b[1] + b[3]) - max(a[1], b[1])
    overlap = max(w, 0) * max(h, 0)
    return overlap / (a[2] * a[3] + b[2] * b[3] - overlap)


def complete(row):
    if row[0] < MIN_AREA or np.isnan(row[5]) or row[5] <= 0:
        return False
    return LEFT_LIMIT + 2 < row[1] and row[1] + row[3] < RIGHT_LIMIT - 2 and 0.6 <= row[3] * BASELINE / row[5] <= 2.2 and 0.8 <= row[4] * BASELINE / row[5] <= 2.2

#### 1. Calibration variants
Each variant is calibrated and then scored in two ways: the reprojection error on the board, and the row difference of SIFT matches on real frames after rectification. The second is the one that matters, because it doesn't use the board at all. `cv2.cornerSubPix` isn't in the course material, so the main notebook leaves it out; this shows what that costs. The negative share is the share of scene matches with a negative disparity, which is physically impossible for a correct rectification.

In [3]:
def corner_variants(path):
    gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    for scale in (1, 1.5, 2, 2.5, 3):
        found, corners = cv2.findChessboardCorners(cv2.resize(gray, None, fx=scale, fy=scale), BOARD)
        if found:
            plain = (corners.reshape(-1, 1, 2) / scale).astype(np.float32)
            criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.01)
            refined = cv2.cornerSubPix(gray, plain.copy(), (5, 5), (-1, -1), criteria).reshape(-1, 2)
            return scale, plain.reshape(-1, 2), refined
    return None


variants_file = os.path.join(RESULTS, 'corner_variants.npy')
if os.path.exists(variants_file):
    found_l, found_r = np.load(variants_file, allow_pickle=True)
else:
    found_l = [corner_variants(p) for p in image_files('0_calib', 'left')]
    found_r = [corner_variants(p) for p in image_files('0_calib', 'right')]
    store = np.empty(2, dtype=object)
    store[0], store[1] = found_l, found_r
    np.save(variants_file, store)

GRID = np.zeros((BOARD[0] * BOARD[1], 3), np.float32)
GRID[:, :2] = np.mgrid[0:BOARD[0], 0:BOARD[1]].T.reshape(-1, 2)


def ordered(corners):
    return corners[::-1] if corners[-1, 0] < corners[0, 0] else corners


def scene_matches(left, right):
    sift = cv2.SIFT_create(4000)
    kp1, des1 = sift.detectAndCompute(cv2.cvtColor(left, cv2.COLOR_BGR2GRAY), None)
    kp2, des2 = sift.detectAndCompute(cv2.cvtColor(right, cv2.COLOR_BGR2GRAY), None)
    good = [m for m, n in cv2.BFMatcher(cv2.NORM_L2).knnMatch(des1, des2, k=2) if m.distance < 0.7 * n.distance]
    return np.float32([kp1[m.queryIdx].pt for m in good]), np.float32([kp2[m.trainIdx].pt for m in good])


def score_calibration(name, one_scale, refine, ransac, flags=0):
    objects_l, images_l, objects_r, images_r, objects_s, stereo_l, stereo_r = [], [], [], [], [], [], []
    for a, b in zip(found_l, found_r):
        for found, objects, images in ((a, objects_l, images_l), (b, objects_r, images_r)):
            if found is not None and not (one_scale and found[0] != 1):
                c = ordered(found[2] if refine else found[1])
                good = cv2.findHomography(GRID[:, :2], c, cv2.RANSAC, 1.0)[1].ravel() == 1 if ransac else np.ones(len(c), bool)
                objects.append(GRID[good])
                images.append(c[good])
        if a is not None and b is not None and not (one_scale and (a[0] != 1 or b[0] != 1)):
            ca, cb = ordered(a[2] if refine else a[1]), ordered(b[2] if refine else b[1])
            good = np.ones(len(ca), bool)
            if ransac:
                good = (cv2.findHomography(GRID[:, :2], ca, cv2.RANSAC, 1.0)[1].ravel() == 1) & (cv2.findHomography(GRID[:, :2], cb, cv2.RANSAC, 1.0)[1].ravel() == 1)
            if np.sum(good) >= 12:
                objects_s.append(GRID[good])
                stereo_l.append(ca[good])
                stereo_r.append(cb[good])
    rms_l, a1, b1, _, _ = cv2.calibrateCamera(objects_l, images_l, IMAGE_SIZE, None, None, flags=flags)
    rms_r, a2, b2, _, _ = cv2.calibrateCamera(objects_r, images_r, IMAGE_SIZE, None, None, flags=flags)
    rms_s, a1, b1, a2, b2, rot, shift, _, _ = cv2.stereoCalibrate(objects_s, stereo_l, stereo_r, a1, b1, a2, b2, IMAGE_SIZE, flags=cv2.CALIB_FIX_INTRINSIC)
    q1, q2, p1, p2, _, _, _ = cv2.stereoRectify(a1, b1, a2, b2, IMAGE_SIZE, rot, shift, alpha=0.15)
    m1 = cv2.initUndistortRectifyMap(a1, b1, q1, p1, IMAGE_SIZE, cv2.CV_32FC1)
    m2 = cv2.initUndistortRectifyMap(a2, b2, q2, p2, IMAGE_SIZE, cv2.CV_32FC1)
    rows, disparities = [], []
    for sequence, frame in (('1_no_obstacle', 10), ('3_curved_bend', 200), ('4_circle', 400)):
        pa, pb = scene_matches(rectified(image_files(sequence, 'left')[frame], 'left', m1), rectified(image_files(sequence, 'right')[frame], 'right', m2))
        rows.append(np.abs(pa[:, 1] - pb[:, 1]))
        disparities.append(pa[:, 0] - pb[:, 0])
    rows, disparities = np.concatenate(rows), np.concatenate(disparities)
    print(f"{name:42s} views {len(stereo_l):2d} | rms L {rms_l:.3f} R {rms_r:.3f} stereo {rms_s:.3f} | scene rows: median {np.median(rows):.2f}, 90% {np.percentile(rows, 90):.2f} | negative {np.mean(disparities < 0):.3f} | baseline {np.linalg.norm(shift):.3f} squares")


start = time.time()
score_calibration('one scale', True, False, False)
score_calibration('several scales', False, False, False)
score_calibration('several scales, cornerSubPix', False, True, False)
score_calibration('several scales, RANSAC (used)', False, False, True)
score_calibration('several scales, cornerSubPix, RANSAC', False, True, True)
score_calibration('  used, k3 fixed at 0', False, False, True, cv2.CALIB_FIX_K3)
score_calibration('  used, k3 and tangential fixed at 0', False, False, True, cv2.CALIB_FIX_K3 | cv2.CALIB_ZERO_TANGENT_DIST)
score_calibration('  used, only k1', False, False, True, cv2.CALIB_FIX_K2 | cv2.CALIB_FIX_K3 | cv2.CALIB_ZERO_TANGENT_DIST)
print("time:", round(time.time() - start, 1), "s")

one scale                                  views 14 | rms L 1.693 R 1.230 stereo 2.380 | scene rows: median 4.63, 90% 13.36 | negative 0.424 | baseline 2.111 squares


several scales                             views 20 | rms L 1.416 R 1.106 stereo 2.316 | scene rows: median 3.60, 90% 19.37 | negative 0.419 | baseline 2.148 squares


several scales, cornerSubPix               views 20 | rms L 1.273 R 1.019 stereo 2.194 | scene rows: median 2.81, 90% 18.06 | negative 0.183 | baseline 2.141 squares


several scales, RANSAC (used)              views 19 | rms L 0.343 R 0.282 stereo 0.430 | scene rows: median 0.61, 90% 1.82 | negative 0.020 | baseline 1.530 squares


several scales, cornerSubPix, RANSAC       views 20 | rms L 0.144 R 0.136 stereo 0.295 | scene rows: median 0.47, 90% 1.84 | negative 0.014 | baseline 1.546 squares


  used, k3 fixed at 0                      views 19 | rms L 0.350 R 0.302 stereo 0.563 | scene rows: median 1.47, 90% 2.93 | negative 0.063 | baseline 1.440 squares


  used, k3 and tangential fixed at 0       views 19 | rms L 0.359 R 0.304 stereo 0.686 | scene rows: median 1.07, 90% 4.53 | negative 0.101 | baseline 1.627 squares


  used, only k1                            views 19 | rms L 0.360 R 0.305 stereo 1.133 | scene rows: median 1.75, 90% 6.68 | negative 0.142 | baseline 1.795 squares
time: 9.7 s


#### 2. How much of the image to keep
`alpha` in `cv2.stereoRectify` sets how much of the original image stays in the rectified one. The focal length shows how much the image is zoomed. The folded share counts output pixels where the map runs backwards: the distortion formula is no longer valid there, and a mirror image of the robot appears.

In [4]:
for alpha in (0, 0.1, 0.15, 0.2, 0.25, 0.4, 0.5, 1):
    q1, q2, p1, p2, _, _, _ = cv2.stereoRectify(K1, d1, K2, d2, IMAGE_SIZE, R, T, alpha=alpha)
    shares = []
    for K, d, q, p in ((K1, d1, q1, p1), (K2, d2, q2, p2)):
        mx, my = cv2.initUndistortRectifyMap(K, d, q, p, IMAGE_SIZE, cv2.CV_32FC1)
        inside = (mx >= 0) & (mx < IMAGE_SIZE[0]) & (my >= 0) & (my < IMAGE_SIZE[1])
        folded = ((np.diff(mx, axis=1)[:-1, :] <= 0) | (np.diff(my, axis=0)[:, :-1] <= 0)) & inside[:-1, :-1]
        shares.append((inside.mean(), folded.mean()))
    print(f"alpha {alpha:<5} focal {p1[0, 0]:6.1f} px | valid: left {shares[0][0]:.3f}, right {shares[1][0]:.3f} | folded: left {shares[0][1]:.4f}, right {shares[1][1]:.4f}")

alpha 0     focal  771.3 px | valid: left 1.000, right 1.000 | folded: left 0.0000, right 0.0000
alpha 0.1   focal  717.0 px | valid: left 0.999, right 0.972 | folded: left 0.0000, right 0.0000
alpha 0.15  focal  689.9 px | valid: left 0.992, right 0.940 | folded: left 0.0000, right 0.0000
alpha 0.2   focal  662.7 px | valid: left 0.971, right 0.906 | folded: left 0.0002, right 0.0000
alpha 0.25  focal  635.6 px | valid: left 0.943, right 0.870 | folded: left 0.0059, right 0.0000
alpha 0.4   focal  554.1 px | valid: left 0.873, right 0.770 | folded: left 0.0543, right 0.0039
alpha 0.5   focal  499.8 px | valid: left 0.824, right 0.729 | folded: left 0.1051, right 0.0379
alpha 1     focal  228.3 px | valid: left 0.313, right 0.311 | folded: left 0.1601, right 0.1588


#### 3. The foreground threshold
Too low a threshold lets noise in, and too high a threshold breaks the dark robot into pieces. For each value, the cell counts on every 5th frame how many frames have a robot-sized blob, how many blobs there are per frame, and the share of the reference box that the largest blob's box covers. When the robot is whole, that share is close to 1.

In [5]:
for threshold in (8, 12, 18, 25, 40):
    seen, counts, covered = 0, [], []
    for sequence in SEQUENCES:
        rows = detections[sequence]
        for i in range(0, len(rows), 5):
            blobs = blob_list(foreground(rectified(image_files(sequence, 'left')[i], 'left'), backgrounds[sequence], threshold))
            counts.append(len(blobs))
            if blobs and blobs[0][0] >= MIN_AREA:
                seen += 1
                if complete(rows[i]):
                    covered.append(blobs[0][3] * blobs[0][4] / (rows[i, 3] * rows[i, 4]))
    print(f"threshold {threshold:2d} | frames with a robot-sized blob {seen} of {len(counts)} | blobs per frame {np.mean(counts):.1f} | largest box / reference box: median {np.median(covered):.2f}, 10% {np.percentile(covered, 10):.2f}")

threshold  8 | frames with a robot-sized blob 267 of 321 | blobs per frame 4.5 | largest box / reference box: median 1.14, 10% 1.08


threshold 12 | frames with a robot-sized blob 263 of 321 | blobs per frame 3.2 | largest box / reference box: median 1.05, 10% 1.02


threshold 18 | frames with a robot-sized blob 256 of 321 | blobs per frame 2.2 | largest box / reference box: median 1.00, 10% 0.68


threshold 25 | frames with a robot-sized blob 251 of 321 | blobs per frame 2.0 | largest box / reference box: median 0.94, 10% 0.59


threshold 40 | frames with a robot-sized blob 217 of 321 | blobs per frame 2.8 | largest box / reference box: median 0.38, 10% 0.12


#### 4. Other detectors from the course
Three simple alternatives are compared with the background subtraction on every 5th frame:
- **Colour** (Class 1): the robot's yellow panels, thresholded in HSV with hue 15 to 35. The cell prints the share of the robot's saturated pixels in that range.
- **Frame difference:** the same operation as background subtraction, but against the previous frame.
- **Optical flow** (Class 2): `cv2.calcOpticalFlowFarneback` at half size, keeping the pixels that move more than 1 px.

A hit means that the detector's box centre lies inside the reference box, and a good box means an IoU of at least 0.5 with it. A false alarm is a detection in a frame where the reference sees no robot.

In [6]:
hues = []
for sequence in SEQUENCES:
    rows = detections[sequence]
    for i in range(0, len(rows), 20):
        if complete(rows[i]):
            _, x, y, w, h = rows[i, :5].astype(int)
            crop = cv2.cvtColor(rectified(image_files(sequence, 'left')[i], 'left')[y:y + h, x:x + w], cv2.COLOR_BGR2HSV)
            hues.append(crop[(crop[:, :, 1] > 120) & (crop[:, :, 2] > 100), 0])
hues = np.concatenate(hues)
low, high = 15, 35
print("saturated pixels on the robot:", len(hues), "| share with hue 15 to 35:", round(np.mean((hues >= low) & (hues <= high)), 3))


def largest(mask):
    blobs = blob_list(cv2.dilate(cv2.erode(mask, None, iterations=1), None, iterations=2))
    return blobs[0][1:] if blobs and blobs[0][0] >= 200 else None


def colour_mask(image, previous):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    return ((hsv[:, :, 0] >= low) & (hsv[:, :, 0] <= high) & (hsv[:, :, 1] > 120) & (hsv[:, :, 2] > 100)).astype(np.uint8) * 255


def difference_mask(image, previous):
    return (np.abs(image.astype(int) - previous.astype(int)).max(axis=2) > 18).astype(np.uint8) * 255


def flow_mask(image, previous):
    a = cv2.resize(cv2.cvtColor(previous, cv2.COLOR_BGR2GRAY), None, fx=0.5, fy=0.5)
    b = cv2.resize(cv2.cvtColor(image, cv2.COLOR_BGR2GRAY), None, fx=0.5, fy=0.5)
    flow = cv2.calcOpticalFlowFarneback(a, b, None, 0.5, 3, 15, 3, 5, 1.5, 0)
    return cv2.resize((np.linalg.norm(flow, axis=2) > 1).astype(np.uint8) * 255, IMAGE_SIZE)


for name, make_mask in (('colour', colour_mask), ('frame difference', difference_mask), ('optical flow', flow_mask)):
    start = time.time()
    hits, good, present, alarms, absent = 0, 0, 0, 0, 0
    per_sequence = {}
    for sequence in SEQUENCES:
        rows, files = detections[sequence], image_files(sequence, 'left')
        seq_hits, seq_present = 0, 0
        for i in range(5, len(rows), 5):
            box = largest(make_mask(rectified(files[i], 'left'), rectified(files[i - 1], 'left')))
            if complete(rows[i]):
                seq_present += 1
                if box is not None:
                    cx, cy = box[0] + box[2] / 2, box[1] + box[3] / 2
                    seq_hits += rows[i, 1] <= cx <= rows[i, 1] + rows[i, 3] and rows[i, 2] <= cy <= rows[i, 2] + rows[i, 4]
                    good += iou(box, rows[i, 1:5]) >= 0.5
            elif rows[i, 0] < MIN_AREA:
                absent += 1
                alarms += box is not None
        hits, present = hits + seq_hits, present + seq_present
        per_sequence[sequence[0]] = round(float(seq_hits / max(seq_present, 1)), 2)
    print(f"{name:17s} hit {hits / present:.3f} ({per_sequence}) | good box {good / present:.3f} | false alarms {alarms} of {absent} | {time.time() - start:.0f} s")

saturated pixels on the robot: 10232 | share with hue 15 to 35: 0.384


colour            hit 0.150 ({'1': 0.15, '2': 0.18, '3': 0.06, '4': 0.19}) | good box 0.000 | false alarms 5 of 64 | 18 s


frame difference  hit 0.628 ({'1': 1.0, '2': 1.0, '3': 0.64, '4': 0.53}) | good box 0.009 | false alarms 3 of 64 | 28 s


optical flow      hit 0.584 ({'1': 1.0, '2': 1.0, '3': 0.59, '4': 0.48}) | good box 0.442 | false alarms 8 of 64 | 32 s


#### 5. A classifier instead of motion: sliding window with PCA and an SVM
Background subtraction can't see a robot that stops, and a live system wouldn't have a median of the whole sequence. The sliding window of Class 8 needs neither. Here it is trained on three sequences and tested on the fourth. The boxes of the main detector are the positive samples, and random crops of the same frames are the negatives. Each window is resized to 40 × 40 pixels, reduced by `PCA`, and scored by an RBF `svm.SVC`. Non-maximum suppression with IoU then removes duplicate boxes. The lecture pairs the SVM with HOG features; this test uses plain pixels, because no course template shows HOG.

In [7]:
PATCH = (40, 40)
SIZES = (40, 56, 80, 112, 160, 224)
rng = np.random.default_rng(0)


def patch_of(gray, box):
    x, y, w, h = [int(v) for v in box]
    return cv2.resize(gray[y:y + h, x:x + w], PATCH).ravel() / 255.0


def training_set(sequences):
    samples, targets = [], []
    for sequence in sequences:
        rows, files = detections[sequence], image_files(sequence, 'left')
        for i in range(0, len(rows), 3):
            if not complete(rows[i]) or rows[i, 2] + rows[i, 4] > IMAGE_SIZE[1]:
                continue
            gray = cv2.cvtColor(rectified(files[i], 'left'), cv2.COLOR_BGR2GRAY)
            samples.append(patch_of(gray, rows[i, 1:5]))
            targets.append(1)
            size = max(rows[i, 3], rows[i, 4])
            for _ in range(3):
                candidate = (rng.integers(0, IMAGE_SIZE[0] - size), rng.integers(300, IMAGE_SIZE[1] - size), size, size)
                if iou(candidate, rows[i, 1:5]) < 0.1:
                    samples.append(patch_of(gray, candidate))
                    targets.append(0)
    return np.array(samples), np.array(targets)


def non_maximum_suppression(boxes, scores, limit=0.5):
    kept = []
    for k in np.argsort(scores)[::-1]:
        if all(iou(boxes[k], boxes[j]) <= limit for j in kept):
            kept.append(k)
    return kept


def sliding_window(gray, pca, clf):
    boxes, patches = [], []
    for size in SIZES:
        step = size // 4
        for y in range(340, IMAGE_SIZE[1] - size, step):
            for x in range(0, IMAGE_SIZE[0] - size, step):
                boxes.append((x, y, size, size))
                patches.append(cv2.resize(gray[y:y + size, x:x + size], PATCH).ravel() / 255.0)
    scores = clf.decision_function(pca.transform(np.array(patches)))
    boxes = np.array(boxes)
    above = scores > 0
    if not above.any():
        return [], []
    boxes, scores = boxes[above], scores[above]
    kept = non_maximum_suppression(boxes, scores)
    return boxes[kept], scores[kept]


for test in ('2_obstacle', '4_circle'):
    start = time.time()
    samples, targets = training_set([s for s in SEQUENCES if s != test])
    pca = PCA(n_components=40, random_state=0)
    clf = svm.SVC(kernel='rbf', C=10, gamma='scale').fit(pca.fit_transform(samples), targets)
    rows, files = detections[test], image_files(test, 'left')
    hits, good, present, extra, alarms, absent = 0, 0, 0, [], 0, 0
    for i in range(0, len(rows), 10):
        found, scores = sliding_window(cv2.cvtColor(rectified(files[i], 'left'), cv2.COLOR_BGR2GRAY), pca, clf)
        if complete(rows[i]):
            present += 1
            extra.append(max(len(found) - 1, 0))
            if len(found):
                b = found[0]
                cx, cy = b[0] + b[2] / 2, b[1] + b[3] / 2
                hits += rows[i, 1] <= cx <= rows[i, 1] + rows[i, 3] and rows[i, 2] <= cy <= rows[i, 2] + rows[i, 4]
                good += iou(b, rows[i, 1:5]) >= 0.5
        elif rows[i, 0] < MIN_AREA:
            absent += 1
            alarms += len(found) > 0
    print(f"test on {test}: {len(samples)} training samples, {np.sum(targets)} positive, explained variance {pca.explained_variance_ratio_.sum():.3f} | hit {hits / present:.3f} | good box {good / present:.3f} | other boxes per frame {np.mean(extra):.1f} | false alarms {alarms} of {absent} | {time.time() - start:.0f} s")

test on 2_obstacle: 1397 training samples, 363 positive, explained variance 0.957 | hit 0.833 | good box 0.333 | other boxes per frame 1.2 | false alarms 9 of 16 | 18 s


test on 4_circle: 618 training samples, 166 positive, explained variance 0.947 | hit 0.462 | good box 0.415 | other boxes per frame 3.5 | false alarms 0 of 0 | 20 s


#### 6. Disparity of the robot
Four ways of measuring the robot's disparity from Classes 2 and 3 are compared on every 2nd frame:
- **StereoBM:** the median of `StereoBM` over the foreground pixels.
- **Box offset:** the difference between the box centres in the left and right image.
- **Template:** `cv2.matchTemplate` of the left box along the same rows of the right image, with a parabola through the peak for sub-pixel accuracy.
- **SIFT:** the median disparity of SIFT matches inside the box, keeping pairs on nearly the same row.

In sequences 1 and 2 the distance is nearly constant, so the frame-to-frame jitter and the spread of the depth measure the noise. Sequence 4 shows the behaviour on the small, distant robot.

In [8]:
def robot_disparities(sequence, i, row):
    left, right = rectified(image_files(sequence, 'left')[i], 'left'), rectified(image_files(sequence, 'right')[i], 'right')
    gray_l, gray_r = cv2.cvtColor(left, cv2.COLOR_BGR2GRAY), cv2.cvtColor(right, cv2.COLOR_BGR2GRAY)
    x, y, w, h = row[1:5].astype(int)
    result = [row[5], np.nan, np.nan, np.nan]
    blobs = blob_list(foreground(right, backgrounds_r[sequence]))
    if blobs:
        result[1] = (x + w / 2) - (blobs[0][1] + blobs[0][3] / 2)
    x0, x1 = max(0, x - 80), min(IMAGE_SIZE[0], x + w + 20)
    scores = cv2.matchTemplate(gray_r[y:y + h, x0:x1], gray_l[y:y + h, x:x + w], cv2.TM_CCOEFF_NORMED).ravel()
    k = int(np.argmax(scores))
    shift = 0.0
    if 0 < k < len(scores) - 1:
        bend = scores[k - 1] - 2 * scores[k] + scores[k + 1]
        shift = 0.5 * (scores[k - 1] - scores[k + 1]) / bend if bend != 0 else 0.0
    result[2] = x - (x0 + k + shift)
    sift = cv2.SIFT_create()
    kp1, des1 = sift.detectAndCompute(gray_l[y:y + h, x:x + w], None)
    kp2, des2 = sift.detectAndCompute(gray_r[y:y + h, x0:x1], None)
    if des1 is not None and des2 is not None and len(kp2) >= 2:
        found = []
        for pair in cv2.BFMatcher(cv2.NORM_L2).knnMatch(des1, des2, k=2):
            if len(pair) == 2 and pair[0].distance < 0.75 * pair[1].distance:
                p, q = kp1[pair[0].queryIdx].pt, kp2[pair[0].trainIdx].pt
                if abs(p[1] - q[1]) < 2 and -2 < (p[0] + x) - (q[0] + x0) < 80:
                    found.append((p[0] + x) - (q[0] + x0))
        if len(found) >= 3:
            result[3] = np.median(found)
    return result


for sequence in ('1_no_obstacle', '2_obstacle', '4_circle'):
    rows = detections[sequence]
    table = np.array([robot_disparities(sequence, i, rows[i]) for i in range(0, len(rows), 2) if complete(rows[i])])
    print(sequence, "frames used:", len(table))
    for k, name in enumerate(('StereoBM', 'box offset', 'template', 'SIFT')):
        d = table[:, k]
        depth = FOCAL * BASELINE / np.where(d > 1, d, np.nan)
        low_z, mid_z, high_z = np.nanpercentile(depth, [10, 50, 90])
        print(f"  {name:11s} measured {np.mean(~np.isnan(d)):.2f} | disparity {np.nanmedian(d):5.1f} px | jitter {np.nanmedian(np.abs(np.diff(d))):.2f} px | depth 10/50/90 %: {low_z:.2f}/{mid_z:.2f}/{high_z:.2f} m | spread {(high_z - low_z) / mid_z:.3f}")

1_no_obstacle frames used: 52
  StereoBM    measured 1.00 | disparity  20.8 px | jitter 0.06 px | depth 10/50/90 %: 3.82/4.06/4.41 m | spread 0.146
  box offset  measured 1.00 | disparity  21.0 px | jitter 1.00 px | depth 10/50/90 %: 3.45/4.02/4.69 m | spread 0.310
  template    measured 1.00 | disparity  19.2 px | jitter 0.24 px | depth 10/50/90 %: 4.20/4.41/15.67 m | spread 2.605
  SIFT        measured 1.00 | disparity  19.1 px | jitter 0.48 px | depth 10/50/90 %: 4.13/4.41/6.18 m | spread 0.464


2_obstacle frames used: 29
  StereoBM    measured 1.00 | disparity  16.6 px | jitter 0.12 px | depth 10/50/90 %: 4.40/5.10/5.44 m | spread 0.204
  box offset  measured 1.00 | disparity  16.0 px | jitter 1.00 px | depth 10/50/90 %: 3.95/4.82/5.82 m | spread 0.389
  template    measured 1.00 | disparity   9.4 px | jitter 0.49 px | depth 10/50/90 %: 4.91/9.00/15.91 m | spread 1.222
  SIFT        measured 1.00 | disparity  15.3 px | jitter 0.48 px | depth 10/50/90 %: 5.13/5.51/15.61 m | spread 1.902


4_circle frames used: 325
  StereoBM    measured 1.00 | disparity   7.9 px | jitter 0.06 px | depth 10/50/90 %: 5.14/10.64/13.78 m | spread 0.812
  box offset  measured 1.00 | disparity   8.0 px | jitter 0.50 px | depth 10/50/90 %: 5.28/10.55/14.07 m | spread 0.833
  template    measured 1.00 | disparity   7.7 px | jitter 0.08 px | depth 10/50/90 %: 5.33/10.98/14.04 m | spread 0.793
  SIFT        measured 1.00 | disparity   7.8 px | jitter 0.09 px | depth 10/50/90 %: 5.29/10.88/13.92 m | spread 0.794
